## Imports

In [1]:
import sqlite3
import glob
import time
import itertools 

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

/var/folders/m8/kq3tbvsx79n47wprd2zwv5hm0000gn/T/ipykernel_87203/1025221560.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


_____

## 1. Data Ingestion 

Read the data from the database.

In [4]:
annotated_dbs = glob.glob(f'./dataset/laca_system/*.db')

print("Found the following dbs : " , annotated_dbs)

synchronous_annotators = ['group_1' , 'group_2'] 
asynchronous_annotators = ['async_1' , 'async_2' , 'async_3'] 
annotators = synchronous_annotators + asynchronous_annotators
annotator2df = {}

for db_path in  annotated_dbs : 

    con = sqlite3.connect(db_path)
    tweet_df = pd.read_sql_query('SELECT * FROM tweet;' , con) 
    theme_df = pd.read_sql_query('SELECT * FROM theme;' , con) 
    theme_df = theme_df.rename(columns={'id' : 'theme_id'})
    merged_df = tweet_df.merge(theme_df , how='left' , on='theme_id')

    for key in annotators : 
        if key in db_path : 
            annotator2df[key] = merged_df

Found the following dbs :  ['./dataset/laca_system/async_1.db', './dataset/laca_system/group_1.db', './dataset/laca_system/async_3.db', './dataset/laca_system/group_2.db', './dataset/laca_system/async_2.db']


### 1.a. Random Sampling rows for manual annotations

Pull 200 samples from both synchronous and asynchronous groups to manually review label quality. Pulling from top 25th percentile is done later.

In [3]:
synchronous_random_samples = []
asynchronous_random_samples = []

for annotator , df in annotator2df.items(): 
    if annotator in synchronous_annotators: 
        synchronous_random_samples.append(df.sample(n=100))
    if annotator in asynchronous_annotators: 
        asynchronous_random_samples.append(df.sample(n=67))

res_df = pd.concat(synchronous_random_samples)
async_df = pd.concat(asynchronous_random_samples)

res_df = res_df[['text' , 'name']].sample(frac=1)
async_df = async_df[['text' , 'name']].sample(frac=1)

# res_df.to_csv('./dataset/generated_samples/laca_sync_sample_all.csv' , index=False, sep='\t')
# async_df.to_csv('./dataset/generated_samples/laca_async_sample_all.csv' , index=False, sep='\t')

_____

## 2. Jaccard Similarity 

Jaccard similarity for two themes is calculated by the union of their documents divided by the intersection of their documents.

In [4]:
results = []

for anno_1 , anno_2 in itertools.permutations(annotators , 2): 
    anno_1_themes = annotator2df[anno_1]['name'].unique()
    anno_2_themes = annotator2df[anno_2]['name'].unique()
    for anno_1_theme , anno_2_theme in itertools.product(anno_1_themes , anno_2_themes): 
        result = {'anno_1' : anno_1 , 
                  'anno_2' : anno_2 , 
                  'anno_1_theme' : anno_1_theme ,
                  'anno_2_theme' : anno_2_theme}
        anno_1_tweet_ids = set(annotator2df[anno_1][annotator2df[anno_1]['name']==anno_1_theme]['tweet_id'])
        anno_2_tweet_ids = set(annotator2df[anno_2][annotator2df[anno_2]['name']==anno_2_theme]['tweet_id'])
        intersection = anno_1_tweet_ids.intersection(anno_2_tweet_ids)
        union = anno_1_tweet_ids.union(anno_2_tweet_ids)
        jaccard_sim = len(intersection) / len(union)
        result['jaccard_sim'] = jaccard_sim
        results.append(result)

jaccard_df = pd.DataFrame(results)

### 2.a. Getting max jaccard similarity for synchronous experiments

In [5]:
max_jacc_sims = []
filtered_df = jaccard_df[(jaccard_df['anno_1'].isin(synchronous_annotators)) 
                         &(jaccard_df['anno_2'].isin(synchronous_annotators))]

for anno_1_theme in filtered_df['anno_1_theme'].unique(): 
    if ('kmeans' not in anno_1_theme.lower()) and ('Unknown' not in anno_1_theme.strip()) and (anno_1_theme != "None"):
        theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
        max_jacc_sims.append(theme_filtered_df.loc[(theme_filtered_df['jaccard_sim'].idxmax())].to_dict())
res_df = pd.DataFrame(max_jacc_sims)

print("Synchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")
print(res_df.count())

Synchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.14
Standard Deviation of Jaccard Similarity: 0.08
anno_1          22
anno_2          22
anno_1_theme    22
anno_2_theme    22
jaccard_sim     22
dtype: int64


In [6]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,sync1_3,sync2_22,GetAVaccine,vaccine_get_a_vaccine,0.237599
1,sync1_3,sync2_22,CovidLiberal,vaccine_politics,0.091247
2,sync1_3,sync2_22,ScrewYouGovernment,vaccine_politics,0.116405
3,sync1_3,sync2_22,CovidDebunking,vaccine_efficacy,0.046923
4,sync1_3,sync2_22,GoodNewsAboutVaccines,online_references,0.045555
5,sync1_3,sync2_22,VaccineAppointmentAvailable,vaccine_rollout,0.225646
6,sync1_3,sync2_22,GotFirstDose,vaccine_first_dose,0.221395
7,sync1_3,sync2_22,JudgingUnvaccinated,vaccine_politics,0.049056
8,sync1_3,sync2_22,ThankYouGovernment,vaccine_rollout,0.026709
9,sync1_3,sync2_22,VaccineSymptomsNegative,vaccine_symptoms,0.186731


### 2.b. Getting max jaccard similarity for asynchronous experiments

In [7]:
max_jacc_sims = []
filtered_df = jaccard_df[(jaccard_df['anno_1'].isin(asynchronous_annotators)) 
                         &(jaccard_df['anno_2'].isin(asynchronous_annotators))]
for anno_1_theme in filtered_df['anno_1_theme'].unique():
    if ('kmeans' not in anno_1_theme.lower()) and  ('Unknown' not in anno_1_theme.strip()) and (anno_1_theme != "None"):
        theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) & 
                                            ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
        max_jacc_sims.append(theme_filtered_df.loc[(theme_filtered_df['jaccard_sim'].idxmax())].to_dict())
res_df = pd.DataFrame(max_jacc_sims)

print("Asynchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")

Asynchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.17
Standard Deviation of Jaccard Similarity: 0.11


In [8]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,alexandra,dj,Receiving_first_dose_of_covid_vaccine,received_vax,0.299751
1,alexandra,alvin,Praising_Government_Leadership,AdvocateForVaccine,0.090649
2,alexandra,alvin,Providing_resources_to_those_who_are_not_invol...,WhereToGetVaccine,0.193657
3,alexandra,dj,Vaccine_accessibility_and_distribution,news_about_vax,0.099928
4,alexandra,dj,Praising_frontline_healthcare_workers,received_vax,0.036316
5,alexandra,dj,Promoting_covid_related_news_articles,news_about_vax,0.119147
6,alexandra,alvin,Information_about_covid_vaccine_availability_a...,WhereToGetVaccine,0.205896
7,alexandra,alvin,Stating_that_the_vaccine_does_not_cause_covid,VaccineEfficacyDenial,0.047455
8,alexandra,alvin,Skepticism_over_the_covid_vaccine,VaccineEfficacyDenial,0.227668
9,alexandra,dj,Criticising_The_President,negative_discourse_around_politicians,0.163701


______

## 3. Centroid Cosine Similarity 

### 3.a. Loading SBERT Vectors 

In [9]:
sbert_vectors = np.load('./dataset/sbert.npy')

### 3.b. Calculating centroids for sync + async experiments

In [10]:
results = []

for annotator, df in annotator2df.items(): 
    cosine_sims = []
    themes = []
    for i , theme in enumerate(df['name'].unique()): 
        if 'kmeans' not in theme.lower() and theme.lower() != "None": 
            result = {'annotator' : annotator}
            ids = df[df['name'] == theme]['tweet_id'].tolist()
            result['theme'] = theme
            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result['theme_centroid'] = theme_centroid
            result['theme_vectors'] = theme_vectors

            dot_prod = np.dot(theme_centroid , theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod , axis=-1)

            norm = np.linalg.norm(theme_vectors , axis=1, keepdims=True)
            cosine_sim = (dot_prod/norm)
            result['cosine_sim']  = cosine_sim 
            results.append(result)


### 3.c. Calculating synchronous centroid cosine similarity

In [11]:
cosine_sim_results = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    anno_1_results = [r for r in results if r['annotator']==anno_1]
    anno_2_results = [r for r in results if r['annotator']==anno_2]
    for anno_1_result in anno_1_results: 
        for anno_2_result in anno_2_results:
            if ('kmeans' not in anno_1_result['theme'].lower()) and  ('kmeans' not in anno_2_result['theme'].lower()) and (anno_2_result['theme'] != "None"): 
                cosine_sim = cosine_similarity(anno_1_result['theme_centroid'] , anno_2_result['theme_centroid'])
                cosine_sim_result = {'anno_1' : anno_1 , 
                                    'anno_2' : anno_2 , 
                                    'anno_1_theme' : anno_1_result['theme'] , 
                                    'anno_2_theme' : anno_2_result['theme'] , 
                                    'cosine_sim' : cosine_sim.squeeze()}
                cosine_sim_results.append(cosine_sim_result)
                
cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [12]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    filtered_df = cosine_sim_df[(cosine_sim_df['anno_1']== anno_1) & (cosine_sim_df['anno_2']== anno_2)]
    for anno_1_theme in filtered_df['anno_1_theme'].unique():
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()) and (anno_1_theme != "None"):
            theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['cosine_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)


print("Synchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}")
print(res_df.count())

Synchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.98
Standard Deviation of Centroid Cosine Similarity: 0.03
anno_1          22
anno_2          22
anno_1_theme    22
anno_2_theme    22
cosine_sim      22
dtype: int64


In [13]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,sync1_3,sync2_22,GetAVaccine,vaccine_get_a_vaccine,0.9967285
1,sync1_3,sync2_22,CovidLiberal,vaccine_politics,0.9907389
2,sync1_3,sync2_22,ScrewYouGovernment,vaccine_politics,0.99006444
3,sync1_3,sync2_22,CovidDebunking,vaccine_efficacy,0.9864078
4,sync1_3,sync2_22,GoodNewsAboutVaccines,vaccine_get_a_vaccine,0.97401154
5,sync1_3,sync2_22,VaccineAppointmentAvailable,vaccine_rollout,0.9866098
6,sync1_3,sync2_22,GotFirstDose,vaccine_first_dose,0.9975873
7,sync1_3,sync2_22,JudgingUnvaccinated,vaccine_efficacy,0.976395
8,sync1_3,sync2_22,ThankYouGovernment,vaccine_rollout,0.98120314
9,sync1_3,sync2_22,VaccineSymptomsNegative,vaccine_symptoms,0.98304176


### 3.d. Calculating asynchronous centroid cosine similarity

In [14]:
cosine_sim_results = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    anno_1_results = [r for r in results if r['annotator']==anno_1]
    anno_2_results = [r for r in results if r['annotator']==anno_2]
    for anno_1_result in anno_1_results : 
        for anno_2_result in anno_2_results :
            if ('kmeans' not in anno_1_result['theme'].lower()) and  ('kmeans' not in anno_2_result['theme'].lower()): 
                cosine_sim = cosine_similarity(anno_1_result['theme_centroid'] , anno_2_result['theme_centroid'])
                cosine_sim_result = {'anno_1' : anno_1 , 
                                    'anno_2' : anno_2 , 
                                    'anno_1_theme' : anno_1_result['theme'] , 
                                    'anno_2_theme' : anno_2_result['theme'] , 
                                    'cosine_sim' : cosine_sim.squeeze()}
                cosine_sim_results.append(cosine_sim_result)
                
cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [22]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    filtered_df = cosine_sim_df[(cosine_sim_df['anno_1']== anno_1) & (cosine_sim_df['anno_2']== anno_2)]
    for anno_1_theme in filtered_df['anno_1_theme'].unique():
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()) and (anno_1_theme != 'None'):
            theme_filtered_df  = filtered_df[(filtered_df['anno_1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno_2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno_2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno_2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['cosine_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)

print("Asynchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}")
print(res_df.count())

Asynchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.98
Standard Deviation of Centroid Cosine Similarity: 0.02
anno_1          76
anno_2          76
anno_1_theme    76
anno_2_theme    76
cosine_sim      76
dtype: int64


In [23]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,alexandra,alvin,Receiving_first_dose_of_covid_vaccine,GotVaccinated,0.9975648
1,alexandra,alvin,Praising_Government_Leadership,AdvocateForVaccine,0.98915243
2,alexandra,alvin,Providing_resources_to_those_who_are_not_invol...,WhereToGetVaccine,0.98367137
3,alexandra,alvin,Vaccine_accessibility_and_distribution,WhereToGetVaccine,0.95742005
4,alexandra,alvin,Praising_frontline_healthcare_workers,GotVaccinated,0.97566044
...,...,...,...,...,...
71,dj,alvin,encourage_getting_vax,AdvocateForVaccine,0.99519503
72,dj,alvin,side_effects_of_vaccine,VaccineSymptomReport,0.98447585
73,dj,alvin,vax_is_ineffective_or_harmful,VaccineEfficacyDenial,0.99181914
74,dj,alvin,getting_attention_of_politician,AdvocateForVaccine,0.9349463


____

## 4. Group Average Cosine Similarity

In [24]:
results = {}
annotators = []

for annotator, df in annotator2df.items():
  results[annotator] = {}
  annotators.append(annotator)
  for i , theme in enumerate(df['name'].unique()):
      if ('kmeans' not in theme.lower()) and ('unknown' not in theme.lower()):
          ids = df[df['name'] == theme]['tweet_id'].tolist()
          theme_vectors = sbert_vectors[ids]
          results[annotator][theme] = theme_vectors

### 4.a. Calculating Group Average Similarities

In [25]:
global_average_sims = []

for (anno1 , anno2) in tqdm(itertools.combinations_with_replacement(annotators, 2)):
  anno1_themes = results[anno1]
  anno2_themes = results[anno2]
  for anno1_theme, anno1_theme_vectors in anno1_themes.items():
    for anno2_theme, anno2_theme_vectors in anno2_themes.items():
      s_time = time.time()
      cosine_sims = cosine_similarity(anno1_theme_vectors , anno2_theme_vectors)
      average_sim = np.average(cosine_sims)
      std_deviation = np.std(cosine_sims)
      global_average_sims.append({'anno1' : anno1 ,
                                  'anno2' : anno2 ,
                                  'anno1_theme' : anno1_theme ,
                                  'anno2_theme' : anno2_theme ,
                                  'average_sim' : average_sim ,
                                  'std_deviation' : std_deviation
                                  })

0it [00:00, ?it/s]

In [26]:
global_average_sims

[{'anno1': 'alexandra',
  'anno2': 'alexandra',
  'anno1_theme': 'Receiving_first_dose_of_covid_vaccine',
  'anno2_theme': 'Receiving_first_dose_of_covid_vaccine',
  'average_sim': 0.48657218,
  'std_deviation': 0.14544544},
 {'anno1': 'alexandra',
  'anno2': 'alexandra',
  'anno1_theme': 'Receiving_first_dose_of_covid_vaccine',
  'anno2_theme': 'None',
  'average_sim': 0.40121832,
  'std_deviation': 0.13162747},
 {'anno1': 'alexandra',
  'anno2': 'alexandra',
  'anno1_theme': 'Receiving_first_dose_of_covid_vaccine',
  'anno2_theme': 'Praising_Government_Leadership',
  'average_sim': 0.40973657,
  'std_deviation': 0.13867436},
 {'anno1': 'alexandra',
  'anno2': 'alexandra',
  'anno1_theme': 'Receiving_first_dose_of_covid_vaccine',
  'anno2_theme': 'Providing_resources_to_those_who_are_not_involved_in_the_covid_debate',
  'average_sim': 0.41812223,
  'std_deviation': 0.13182002},
 {'anno1': 'alexandra',
  'anno2': 'alexandra',
  'anno1_theme': 'Receiving_first_dose_of_covid_vaccine',
  

In [27]:
global_average_df = pd.DataFrame(global_average_sims)

### 4.b. Calculating Synchronous Group Average Cosine Similarities

In [29]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(synchronous_annotators , 2):
    filtered_df = global_average_df[(global_average_df['anno1'] == anno_1)&
                                    (global_average_df['anno2'] == anno_2)]
    for anno_1_theme in filtered_df['anno1_theme'].unique(): 
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()):
            theme_filtered_df  = filtered_df[(filtered_df['anno1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['average_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)

print("Synchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}")

print(res_df.count())

Synchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.44
Standard Deviation of Group Cosine Similarity: 0.03
anno1            16
anno2            16
anno1_theme      16
anno2_theme      16
average_sim      16
std_deviation    16
dtype: int64


### 4.c. Calculating Asynchronous Group Average Cosine Similarities

In [31]:
max_cosine_sims = []

for anno_1 , anno_2 in itertools.permutations(asynchronous_annotators , 2):
    filtered_df = global_average_df[(global_average_df['anno1'] == anno_1)&
                                    (global_average_df['anno2'] == anno_2)]
    for anno_1_theme in filtered_df['anno1_theme'].unique(): 
        if ('kmeans' not in anno_1_theme.lower()) and ('unknown' not in anno_1_theme.lower()):
            theme_filtered_df  = filtered_df[(filtered_df['anno1_theme'] == anno_1_theme) &
                                            ~(filtered_df['anno2_theme'].str.contains('None')) & 
                                              ~(filtered_df['anno2_theme'].str.contains('Kmeans')) &
                                              ~(filtered_df['anno2_theme'].str.contains('Unknown'))]
            max_cosine_sims.append(theme_filtered_df.loc[(theme_filtered_df['average_sim'].idxmax())].to_dict())
 
res_df = pd.DataFrame(max_cosine_sims)


print("Asynchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}")

print(res_df.count())

Asynchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.45
Standard Deviation of Group Cosine Similarity: 0.03
anno1            46
anno2            46
anno1_theme      46
anno2_theme      46
average_sim      46
std_deviation    46
dtype: int64


____

## 5. Getting top 25th percentile of closest vectors to each centroid

In [32]:
results = []

for annotator, df in annotator2df.items(): 
    cosine_sims = []
    themes = []
    for i , theme in enumerate(df['name'].unique()): 
        if ('kmeans' not in theme.lower()) and ('unknown' not in theme.lower()) and (theme != "None"): 
            result = {'annotator' : annotator}
            ids = df[df['name'] == theme]['tweet_id'].tolist()
            result['theme'] = theme
            result['tweets'] = df[df['name'] == theme]['text'].tolist()

            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result['theme_centroid'] = theme_centroid
            result['theme_vectors'] = theme_vectors

            dot_prod = np.dot(theme_centroid , theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod , axis=-1)

            norm = np.linalg.norm(theme_vectors , axis=1, keepdims=True)
            cosine_sim = (dot_prod/norm)
            result['cosine_sim']  = cosine_sim.squeeze() 
            results.append(result)

In [ ]:
top_25_results = {'top_25_tweets':[],
                  'top_25_vectors' : [], 
                  'theme' : [], 
                  'annotator' : []}

for result in results: 

    m = np.percentile(result['cosine_sim'] , 25)
    n = np.percentile(result['cosine_sim'] , 25)
    top_25_indices = np.where(result['cosine_sim']<m)
    # top_25_indices = np.where(np.logical_and(result['cosine_sim']>n , result['cosine_sim']<m))
    top_25_vectors = result['theme_vectors'][top_25_indices]
    top_25_tweets = np.array(result['tweets'])[top_25_indices]
    annotator=result['annotator']
    theme=result['theme']

    top_25_results['top_25_tweets'].extend(top_25_tweets)
    top_25_results['top_25_vectors'].extend(top_25_vectors )
    top_25_results['theme'].extend([theme]*top_25_tweets.shape[0])
    top_25_results['annotator'].extend([annotator]*top_25_tweets.shape[0])


top_25_df = pd.DataFrame(top_25_results, columns=['theme' , 'annotator' , 'top_25_tweets'])

### 4.a. Random Sampling rows for manual annotations

In [ ]:
sync_top_25 = top_25_df[top_25_df['annotator'].isin(synchronous_annotators)].sample(n=1000)[['theme' , 'top_25_tweets']]
async_top_25 = top_25_df[top_25_df['annotator'].isin(asynchronous_annotators)].sample(n=1000)[['theme' , 'top_25_tweets']]

sync_top_25.to_csv('./dataset/generated_samples/laca_sync_sample_0to25.csv' , index=False, sep='\t')
async_top_25.to_csv('./dataset/generated_samples/laca_async_sample_0to25.csv' , index=False, sep='\t')

____

## 5. Calculating Intra- and Inter-cluster Similarity

In [35]:
top_25_results = {}

for result in results: 

    top_25_results[result['theme']] = {}

    m = np.percentile(result['cosine_sim'] , 75)
    top_25_indices = np.where(result['cosine_sim']>m)
    top_25_vectors = result['theme_vectors'][top_25_indices]
    top_25_tweets = np.array(result['tweets'])[top_25_indices]
    annotator=result['annotator']
    theme=result['theme']

    top_25_results[theme]['top_25_tweets'] = (top_25_tweets)
    top_25_results[theme]['top_25_vectors'] = (top_25_vectors )
    top_25_results[theme]['annotator'] = (annotator)

### 5.a. Calculating similarities for top 25th percentile clusters

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

top_25_average_sims = []

for anno_1_theme , anno1_top_25 in tqdm(top_25_results.items()): 
    for anno_2_theme , anno2_top_25 in top_25_results.items(): 

        anno1_top_25_vectors = anno1_top_25['top_25_vectors']
        anno2_top_25_vectors = anno2_top_25['top_25_vectors']

        cosine_sims = cosine_similarity(anno1_top_25_vectors , anno2_top_25_vectors)
        average_sim = np.average(cosine_sims)
        std_deviation = np.std(cosine_sims)

        top_25_average_sims.append({'anno_1' : anno1_top_25['annotator'] , 
                                   'anno_2' : anno2_top_25['annotator'], 
                                   'anno_1_theme' : anno_1_theme,
                                   'anno_2_theme' : anno_2_theme,
                                   'average_sim' : average_sim , 
                                   'std_dev_sim' : std_deviation})


  0%|          | 0/60 [00:00<?, ?it/s]

In [37]:
top_25_average_df = pd.DataFrame(top_25_average_sims)
top_25_average_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,average_sim,std_dev_sim
0,alexandra,alexandra,Receiving_first_dose_of_covid_vaccine,Receiving_first_dose_of_covid_vaccine,0.719753,0.071794
1,alexandra,alexandra,Receiving_first_dose_of_covid_vaccine,Praising_Government_Leadership,0.567607,0.084653
2,alexandra,alexandra,Receiving_first_dose_of_covid_vaccine,Providing_resources_to_those_who_are_not_invol...,0.564993,0.074919
3,alexandra,alexandra,Receiving_first_dose_of_covid_vaccine,Vaccine_accessibility_and_distribution,0.537902,0.074487
4,alexandra,alexandra,Receiving_first_dose_of_covid_vaccine,Praising_frontline_healthcare_workers,0.647123,0.074139
...,...,...,...,...,...,...
3595,sync2_22,sync2_22,online_references,vaccine_rollout,0.522907,0.099415
3596,sync2_22,sync2_22,online_references,vaccine_efficacy,0.477832,0.120104
3597,sync2_22,sync2_22,online_references,vaccine_first_dose,0.482601,0.116718
3598,sync2_22,sync2_22,online_references,vaccine_symptoms,0.420470,0.119692


In [38]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']==top_25_average_df['anno_2_theme'])] 

### 5.b. Intra- and Inter-theme similarities for top 25th percentile subsets in synchronous and asynchronous experiments

In [39]:
print(f"Synchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% intra theme similarity average: 0.63
Synchronous top 25% intra theme similarity standard deviation: 0.07
Asynchronous top 25% intra theme similarity average: 0.63
Asynchronous top 25% intra theme similarity standard deviation: 0.05


In [40]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']!=top_25_average_df['anno_2_theme'])] 

In [41]:
print(f"Synchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% inter theme similarity average: 0.55
Synchronous top 25% inter theme similarity standard deviation: 0.05
Asynchronous top 25% inter theme similarity average: 0.54
Asynchronous top 25% inter theme similarity standard deviation: 0.05


### 5.c. Intra- and Inter-theme similarities for whole set in synchronous and asynchronous experiments

In [42]:
global_average_df = pd.DataFrame(global_average_sims)

intra_global_average_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']==global_average_df['anno2_theme'])] 


print(f"Synchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global intra theme similarity average: 0.44
Synchronous global intra theme similarity standard deviation: 0.06
Asynchronous global intra theme similarity average: 0.43
Asynchronous global intra theme similarity standard deviation: 0.05


In [43]:
inter_global_avg_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']!=global_average_df['anno2_theme'])] 

print(f"Synchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global inter theme similarity average: 0.40
Synchronous global inter theme similarity standard deviation: 0.04
Asynchronous global inter theme similarity average: 0.39
Asynchronous global inter theme similarity standard deviation: 0.04


## 6. Jaccard Similarity Heatmap — Synchronous (sync1_3 vs sync2_22)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

exclude_patterns = ['kmeans', 'unknown', 'none']

def should_keep(theme):
    return not any(p in theme.lower() for p in exclude_patterns)

# Filter jaccard_df for sync1_3 -> sync2_22 direction
jacc_sync = jaccard_df[
    (jaccard_df['anno_1'] == 'sync1_3') & (jaccard_df['anno_2'] == 'sync2_22')
].copy()
jacc_sync = jacc_sync[jacc_sync['anno_1_theme'].apply(should_keep) & jacc_sync['anno_2_theme'].apply(should_keep)]

jacc_pivot = jacc_sync.pivot(index='anno_1_theme', columns='anno_2_theme', values='jaccard_sim')

plt.figure(figsize=(12, 10))
sns.heatmap(jacc_pivot, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            xticklabels=True, yticklabels=True)
plt.title('Jaccard Similarity — Synchronous (sync1_3 vs sync2_22)')
plt.xlabel('sync2_22 themes')
plt.ylabel('sync1_3 themes')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()